# Install and load both scispaCy models

In [2]:
import spacy

nlp_sci = spacy.load("en_core_sci_md")
nlp_bc5cdr = spacy.load("en_ner_bc5cdr_md")

print("Both models loaded")
print(f"en_core_sci_md labels: {nlp_sci.get_pipe('ner').labels}")
print(f"en_ner_bc5cdr_md labels: {nlp_bc5cdr.get_pipe('ner').labels}")

c:\Users\DELL\Desktop\medrag\venv\Lib\site-packages\spacy\language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


Both models loaded
en_core_sci_md labels: ('ENTITY',)
en_ner_bc5cdr_md labels: ('CHEMICAL', 'DISEASE')


# Run both models on a real WHO chunk, compare what each extracts

In [3]:
sample_text = (
    "Metformin is recommended as first-line pharmacological treatment for type 2 diabetes. "
    "In patients with hypertension, ACE inhibitors such as ramipril reduce cardiovascular risk. "
    "A daily dose of 500mg metformin may be increased gradually to minimize gastrointestinal side effects."
)

print("=== en_core_sci_md ===")
doc_sci = nlp_sci(sample_text)
for ent in doc_sci.ents:
    print(f"  '{ent.text}'  [{ent.label_}]")

print("\n=== en_ner_bc5cdr_md ===")
doc_bc5cdr = nlp_bc5cdr(sample_text)
for ent in doc_bc5cdr.ents:
    print(f"  '{ent.text}'  [{ent.label_}]")

=== en_core_sci_md ===
  'Metformin'  [ENTITY]
  'pharmacological treatment'  [ENTITY]
  'type 2 diabetes'  [ENTITY]
  'patients'  [ENTITY]
  'hypertension'  [ENTITY]
  'ACE inhibitors'  [ENTITY]
  'ramipril'  [ENTITY]
  'cardiovascular risk'  [ENTITY]
  'daily'  [ENTITY]
  'dose'  [ENTITY]
  'metformin'  [ENTITY]
  'increased'  [ENTITY]
  'gastrointestinal side'  [ENTITY]

=== en_ner_bc5cdr_md ===
  'Metformin'  [CHEMICAL]
  'diabetes'  [DISEASE]
  'hypertension'  [DISEASE]
  'ACE inhibitors'  [CHEMICAL]
  'ramipril'  [CHEMICAL]
  'metformin'  [CHEMICAL]


# Build a combined extraction function (NER + dosage regex)

In [4]:
import re

DOSAGE_PATTERN = re.compile(
    r"\b\d+(?:\.\d+)?\s*(?:mg|mcg|g|ml|mL|IU|units?)\b(?:/(?:day|dose|kg|mL))?",
    re.IGNORECASE,
)

def extract_medical_entities(text: str) -> dict:
    """Extract CHEMICAL/DISEASE entities via bc5cdr, plus dosage mentions
    via regex (a trained NER model isn't needed for numeric dosage
    patterns, and bc5cdr wasn't trained to recognize them anyway)."""
    doc = nlp_bc5cdr(text)
    chemicals = [ent.text for ent in doc.ents if ent.label_ == "CHEMICAL"]
    diseases = [ent.text for ent in doc.ents if ent.label_ == "DISEASE"]
    dosages = [m.group() for m in DOSAGE_PATTERN.finditer(text)]

    return {
        "chemicals": chemicals,
        "diseases": diseases,
        "dosages": dosages,
    }

result = extract_medical_entities(sample_text)
print(result)

{'chemicals': ['Metformin', 'ACE inhibitors', 'ramipril', 'metformin'], 'diseases': ['diabetes', 'hypertension'], 'dosages': ['500mg']}


# Test on a real chunk from the corpus

In [5]:
import sys, os
from pathlib import Path
import logging

def find_project_root(marker="backend", start=None):
    current = Path(start or os.getcwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(f"Could not find a '{marker}' folder above {current}")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "backend"))
sys.path.insert(0, str(PROJECT_ROOT / "backend" / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "backend" / "scripts"))

logging.basicConfig(level=logging.INFO, format="%(message)s")

from medrag.processing.storage import load_chunks

CHUNKS_DIR = str(PROJECT_ROOT / "data" / "processed" / "chunks")

# grab a few real hypertension WHO chunks
sample_chunks = list(load_chunks(source="who", topic="hypertension", output_dir=CHUNKS_DIR))[:5]

for chunk in sample_chunks:
    result = extract_medical_entities(chunk.raw_text)
    print(f"chunk_id={chunk.chunk_id}")
    print(f"  text preview: {chunk.raw_text[:120]}...")
    print(f"  chemicals: {result['chemicals']}")
    print(f"  diseases: {result['diseases']}")
    print(f"  dosages: {result['dosages']}")
    print()

chunk_id=hypertension_who_text_0
  text preview: Guideline
for the
pharmacological
treatment of
hypertension
in adults
Guideline for the pharmacological treatment of hyp...
  chemicals: ['CC']
  diseases: ['hypertension', 'hypertension']
  dosages: []

chunk_id=hypertension_who_text_1
  text preview: Any mediation relating to disputes arising under the licence shall be conducted in accordance with the
mediation rules o...
  chemicals: ['CC', 'CIP', 'CIP']
  diseases: ['hypertension']
  dosages: []

chunk_id=hypertension_who_text_2
  text preview: The designations employed and the presentation of the material in this publication
do not imply the expression of any op...
  chemicals: []
  diseases: ['damages']
  dosages: []

chunk_id=hypertension_who_text_3
  text preview: Contents
Acknowledgements v
Acronyms and abbreviations vi
Executive summary vii
1 Introduction 1
2 Method for developing...
  chemicals: []
  diseases: ['Cardiovascular disease', 'Hypertension', 'hypertension', 'hyperte

# Test on later chunks (past front matter, into real content)

In [6]:
# skip the first ~15 chunks (front matter/ToC), sample from real clinical content
sample_chunks_deep = list(load_chunks(source="who", topic="hypertension", output_dir=CHUNKS_DIR))[20:25]

for chunk in sample_chunks_deep:
    result = extract_medical_entities(chunk.raw_text)
    print(f"chunk_id={chunk.chunk_id}")
    print(f"  text preview: {chunk.raw_text[:120]}...")
    print(f"  chemicals: {result['chemicals']}")
    print(f"  diseases: {result['diseases']}")
    print(f"  dosages: {result['dosages']}")
    print()

chunk_id=hypertension_who_text_20
  text preview: Members of the GDG, with the help of the methodologist, developed evidence profiles to summarize
relative and absolute e...
  chemicals: []
  diseases: []
  dosages: []

chunk_id=hypertension_who_text_21
  text preview: A weak or conditional recommendation is one for which the GDG concluded that the desirable
effects of adhering to the re...
  chemicals: []
  diseases: []
  dosages: []

chunk_id=hypertension_who_text_22
  text preview: 2.7 Funding
The development of this guideline was financially supported by the US Centers for Disease Control and
Preven...
  chemicals: ['140', 'creatinine']
  diseases: ['diabetes', 'DM', 'coronary artery disease', 'CAD', 'stroke', 'stroke', 'myocardial infarction', 'heart failure', 'deaths']
  dosages: []

chunk_id=hypertension_who_text_23
  text preview: The benefits were a
reduction in severe events with significant morbidity and mortality whereas the harms were mostly no...
  chemicals: []
  disease

# Run extraction across the full hypertension topic, tally entity frequency

In [7]:
from collections import Counter

all_hypertension_chunks = list(load_chunks(source="who", topic="hypertension", output_dir=CHUNKS_DIR))
print(f"Total chunks: {len(all_hypertension_chunks)}")

chemical_chunk_counts = Counter()
disease_chunk_counts = Counter()

for chunk in all_hypertension_chunks:
    result = extract_medical_entities(chunk.raw_text)
    for chem in set(result["chemicals"]):  # set() so repeats within one chunk count once
        chemical_chunk_counts[chem] += 1
    for dis in set(result["diseases"]):
        disease_chunk_counts[dis] += 1

print(f"\nUnique chemicals found: {len(chemical_chunk_counts)}")
print(f"Unique diseases found: {len(disease_chunk_counts)}")

print("\n=== Chemicals appearing in exactly 1 chunk (bottom of distribution) ===")
singleton_chemicals = [term for term, count in chemical_chunk_counts.items() if count == 1]
print(singleton_chemicals[:30])

print(f"\n=== Chemicals appearing in 5+ chunks (top of distribution) ===")
frequent_chemicals = [(term, count) for term, count in chemical_chunk_counts.most_common(20)]
print(frequent_chemicals)

Total chunks: 146

Unique chemicals found: 106
Unique diseases found: 130

=== Chemicals appearing in exactly 1 chunk (bottom of distribution) ===
['CIP', 'RAAS renin-angiotensin-aldosterone', 'ARB angiotensin-II-receptor', 'ACEi angiotensin-converting', 'alcohol', 'Q9', 'to-large', 'low-dose thiazide', 'thiazides', 'high-dose', 'HTN', 'candesartan', 'low-dose', 'up-titration', 'olmesartan', 'valsartan/amlodipine', 'AUD', 'angiotensin II', 'aldosterone', 'NCT04366050', 'Ramipril', 'methyldopa', 'hydralazine', 'labetalol', 'spironolactone', 'anti-androgen', 'NOITACILBUP', 'HCRAESER', 'ACE/ARB', 'dihydropyridine CCB']

=== Chemicals appearing in 5+ chunks (top of distribution) ===
[('ARB', 14), ('K', 14), ('ACEi', 11), ('ARBs', 11), ('CCBs', 10), ('thiazide', 8), ('beta-blockers', 7), ('DM', 7), ('thiazide-like', 7), ('CCB', 6), ('calcium', 5), ('creatinine', 5), ('chlorthalidone', 5), ('ACEi/ARB', 5), ('CC', 4), ('glucose', 4), ('JA', 4), ('PEN', 3), ('angiotensin', 3), ('lisinopril', 3

# Build and test a targeted noise filter

In [8]:
import re

TRIAL_ID_PATTERN = re.compile(r"^NCT\d+$", re.IGNORECASE)
NON_ALPHA_JUNK_PATTERN = re.compile(r"^[a-z]*$", re.IGNORECASE)  # placeholder, refined below

def is_likely_noise(entity_text: str) -> bool:
    """Filter concrete, observed noise patterns rather than a blunt
    frequency threshold (which was tested and found to conflate real
    rare drug names with genuine garbage at the same frequency)."""
    text = entity_text.strip()

    if TRIAL_ID_PATTERN.match(text):
        return True

    # reversed-text extraction artifacts (found: NOITACILBUP, HCRAESER)
    if text[::-1].lower() in {"publication", "research", "copyright", "reserved"}:
        return True

    # bare numbers or number-only tokens misfiring as CHEMICAL (e.g. "140")
    if re.fullmatch(r"\d+(\.\d+)?", text):
        return True

    return False


# re-test against the full hypertension singleton list
filtered_chemicals = {term: count for term, count in chemical_chunk_counts.items() if not is_likely_noise(term)}
removed = {term: count for term, count in chemical_chunk_counts.items() if is_likely_noise(term)}

print(f"Removed as noise ({len(removed)}): {list(removed.keys())}")
print(f"\nRemaining unique chemicals: {len(filtered_chemicals)}")

Removed as noise (4): ['140', 'NCT04366050', 'NOITACILBUP', 'HCRAESER']

Remaining unique chemicals: 102


# Wrap into a clean extraction+filter function, test full pipeline once more

In [9]:
def extract_medical_entities_filtered(text: str) -> dict:
    """Extract CHEMICAL/DISEASE entities via bc5cdr + dosage via regex,
    then drop entities matching known, concrete noise patterns (clinical
    trial IDs, reversed-text extraction artifacts, bare numbers). Does
    NOT attempt to filter all noise — short ambiguous abbreviations
    (e.g. misfires like 'CIP', 'Q9') are a known, documented remaining
    limitation, since a broader filter risks dropping legitimate short
    drug abbreviations (ARB, ACEi, CCB, DM, HTN)."""
    raw = extract_medical_entities(text)
    return {
        "chemicals": [e for e in raw["chemicals"] if not is_likely_noise(e)],
        "diseases": [e for e in raw["diseases"] if not is_likely_noise(e)],
        "dosages": raw["dosages"],
    }

# re-run on the earlier front-matter chunk that had the worst noise
test_chunk = all_hypertension_chunks[4]  # hypertension_who_text_4, had HHS/A3.1 noise
result = extract_medical_entities_filtered(test_chunk.raw_text)
print(f"chunk_id={test_chunk.chunk_id}")
print(result)

chunk_id=hypertension_who_text_4
{'chemicals': [], 'diseases': ['hypertension', 'A3.1', 'hypertension', 'Hepatitis', 'HHS'], 'dosages': []}
